### installation

uv add langgraph

uv add langchain[google-genai]

In [9]:
import os 
from dotenv import load_dotenv

load_dotenv()
api_key= os.getenv("GEMINI_API_KEY")

In [19]:
from langchain.chat_models import init_chat_model

gemini_model = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai",api_key=api_key)
gemini_model.invoke("Hello, how are you?")

AIMessage(content="Hello! I'm doing well, thank you for asking. I'm a large language model, so I don't experience feelings in the way humans do, but I'm functioning optimally and ready to help you.\n\nHow are you doing today? What can I assist you with?", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': []}, id='run--c00b81cf-71d1-434a-97e7-0ffc83c3f9d2-0', usage_metadata={'input_tokens': 7, 'output_tokens': 60, 'total_tokens': 67, 'input_token_details': {'cache_read': 0}})

In [21]:
from langgraph.prebuilt import create_react_agent

def get_weather(city: str) -> str:
    """ Get weather for a given city """  
    return f"It's always sunny in {city}!"

agent = create_react_agent(
    model =gemini_model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant that can get weather for a given city.",
    api_key=api_key
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='48d62e90-ef46-4b25-84a7-d266368dc0cc'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "sf"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': []}, id='run--7cf5e00b-ecb5-4532-b1d4-04ab3f9e8782-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'sf'}, 'id': 'cb98f0aa-95ac-4e43-b3c9-54163f563a85', 'type': 'tool_call'}], usage_metadata={'input_tokens': 45, 'output_tokens': 15, 'total_tokens': 60, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="It's always sunny in sf!", name='get_weather', id='a9513ae3-dfa4-4ec0-b9cc-e72f60ca0043', tool_call_id='cb98f0aa-95ac-4e43-b3c9-54163f563a85'),
  AIMessage(content="The weather in sf is: It's always sunny in sf!", additional_kwargs={}, resp

# **Add memory**

In [35]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

def get_weather(city: str) -> str:
    """ Get weather for a given city """  
    return f"It's always sunny in {city}!"

agent= create_react_agent(
    model=gemini_model,
    tools=[get_weather],
    checkpointer=checkpointer,
    system_prompt="You are a helpful assistant that can get weather for a given city.",
)

# Run the agent
config = {"configurable": {"thread_id": "1"}}
bd_response = agent.invoke(
    {"messages":[{"role":"user", "content": "Hi, My name is Talha and live on Dhaka. Can you tell me what is the weather in Dhaka?"}]},
    config
)


In [36]:
bd_response["messages"][-1].content

"Hello Talha! It's always sunny in Dhaka!"

In [37]:
ny_response = agent.invoke(
    {"messages": [{"role": "user", "content": "Can you tell me my name?"}]},
    config
)

ny_response["messages"][-1].content

'Your name is Talha.'

## **Configure structured output**

In [39]:
from pydantic import BaseModel
from langgraph.prebuilt import create_react_agent

class WeatherResponse(BaseModel):
    conditions:str
    is_sunny:bool
    description:str  

agent = create_react_agent(
    model=gemini_model,
    tools=[get_weather],
    response_format=WeatherResponse
)

response = agent.invoke({"messages":[{"role":"user","content":"what is the weather in dhaka?"}]})

In [41]:
response["structured_response"]

WeatherResponse(conditions='sunny', is_sunny=True, description="It's always sunny in dhaka!")